In [ ]:
#!/usr/bin/env python3
"""
Project 13 -- G0a-v2 OPTIONAL robustness check (Kaggle handoff)
True full-cis-window aggregate cis signal from eQTLGen's FULL NOMINAL file
(every tested cis-SNP-gene pair genome-wide, not just FDR<0.05 significant
ones -- the literal gold-standard instrument the G0a-v2 task envisioned).

Why this is a Kaggle script, not a sandbox one
------------------------------------------------
The file is 3.88GB gzipped (eqtlgen.org "Full cis-eQTL summary statistics",
2019-12-11-cis-eQTLsFDR-ProbeLevel-CohortInfoRemoved-BonferroniAdded.txt.gz),
~100M+ SNP-gene rows (19,250 genes x every cis-SNP within 1Mb of gene center,
>=2 cohorts). The dev sandbox this project normally runs in has (a) no
persistent background processes between tool calls (confirmed empirically:
a nohup'd background job is killed by bwrap's --die-with-parent the moment
the enclosing tool call returns) and (b) a hard 45s-per-call ceiling -- both
make streaming a file this size impractical there. Same class of problem as
the 1M-scBloodNL raw count matrices (STATE.md 2026-07-15b/h), same fix:
self-contained Kaggle script, network + no call-time constraints there.

What this computes
-------------------
For every gene in the file: n_snps_tested (ALL rows, not just significant),
sum(Zscore^2) over ALL of them (a real cis-h^2-style burden statistic -- the
sum of squared Z-scores across every tested cis-SNP approximates the total
cis-explainable variance under a simple additive/independence assumption;
imperfect given LD between nearby SNPvs but a materially better estimator
than either v1's lead-SNP-only score or v2's significant-only aggregate,
since it uses the COMPLETE tested cis-window, not just the SNPs that
happened to clear a significance threshold).

This is offered as a robustness re-check of G0a-v2's PRIMARY per-gene
cross-cohort criterion (data/processed/g0a_v2_cis_handle_report.json,
primary_per_gene_cross_cohort, r=0.309, missed the 0.5 bar) -- if the
per-gene resolution rescues with the full-window estimator, that upgrades
G0a's clearance from GENE-CLASS to PER-GENE; if not, it corroborates the
v2 gene-class-only verdict with a strictly better instrument. Does NOT
gate or reopen the already-delivered G0a-v2 verdict on its own -- report
back to Manager/Brian either way.

Logic validated in-sandbox on synthetic data before this Kaggle handoff
(see `_selftest()` below, run automatically when SMOKE_TEST=True), same
discipline as every other kaggle_*.py in this repo.

Output
------
    eqtlgen_full_nominal_cis_h2.parquet   (per-gene: gene_id, n_snps_tested,
                                            sum_z2, max_z2)
    eqtlgen_full_nominal_cis_h2_report.json

Usage
-----
    Kaggle: set PHASE_SMOKE_TEST=0 as an env var, run this script (needs
    internet + a dataset/output directory Kaggle provides by default).
    Bring the two output files back into data/processed/ for the Engineer
    to rerun G0a-v2's primary/comparative checks against this score.
"""
from __future__ import annotations
import io
import json
import os
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd

IS_KAGGLE = os.path.exists("/kaggle/input")
OUT_DIR = Path("/kaggle/working") if IS_KAGGLE else Path(".")
SMOKE_TEST = False

URL = ("https://download.gcc.rug.nl/downloads/eqtlgen/cis-eqtl/"
       "2019-12-11-cis-eQTLsFDR-ProbeLevel-CohortInfoRemoved-BonferroniAdded.txt.gz")
LOCAL_GZ = OUT_DIR / "eqtlgen_full_nominal_cis.txt.gz"
OUT_PARQUET = OUT_DIR / "eqtlgen_full_nominal_cis_h2.parquet"
OUT_REPORT = OUT_DIR / "eqtlgen_full_nominal_cis_h2_report.json"

CHUNKSIZE = 2_000_000  # rows per pandas chunk while streaming the gzip


def fetch():
    if LOCAL_GZ.exists():
        print(f"[fetch] {LOCAL_GZ} already present, skipping download")
        return
    print(f"[fetch] downloading {URL} (3.88GB) ...")
    urllib.request.urlretrieve(URL, LOCAL_GZ)
    print(f"[fetch] done, {LOCAL_GZ.stat().st_size} bytes")


def stream_aggregate(path) -> pd.DataFrame:
    """Stream the (possibly huge) tab-separated file in chunks, accumulating
    per-gene n_snps_tested / sum(Zscore^2) / max(Zscore^2) without ever
    holding the full file in memory."""
    sums: dict[str, float] = {}
    counts: dict[str, int] = {}
    maxes: dict[str, float] = {}

    reader = pd.read_csv(path, sep="\t", usecols=["Zscore", "Gene"],
                          compression="gzip", chunksize=CHUNKSIZE)
    total_rows = 0
    for i, chunk in enumerate(reader):
        z2 = chunk["Zscore"].astype(float).values ** 2
        genes = chunk["Gene"].values
        # vectorized-ish accumulation via pandas groupby per chunk, merged into dicts
        g = pd.DataFrame({"gene": genes, "z2": z2}).groupby("gene")["z2"].agg(["sum", "count", "max"])
        for gene, row in g.iterrows():
            sums[gene] = sums.get(gene, 0.0) + row["sum"]
            counts[gene] = counts.get(gene, 0) + int(row["count"])
            maxes[gene] = max(maxes.get(gene, 0.0), row["max"])
        total_rows += len(chunk)
        if i % 10 == 0:
            print(f"[stream] processed {total_rows:,} rows so far, {len(sums):,} genes seen")

    out = pd.DataFrame({
        "gene_id": list(sums.keys()),
        "n_snps_tested": [counts[g] for g in sums],
        "sum_z2": [sums[g] for g in sums],
        "max_z2": [maxes[g] for g in sums],
    })
    print(f"[stream] TOTAL {total_rows:,} rows -> {len(out):,} genes")
    return out


def _selftest():
    """Validate stream_aggregate's arithmetic against a known-answer synthetic
    file before trusting it on the real 3.88GB download -- same discipline
    as kaggle_1m_scbloodnl_de.py's stream_aggregate self-test."""
    rng = np.random.default_rng(0)
    n = 500_000
    genes = rng.choice([f"ENSG{i:05d}" for i in range(200)], size=n)
    z = rng.normal(0, 1, size=n)
    df = pd.DataFrame({"Gene": genes, "Zscore": z})
    tmp = OUT_DIR / "_selftest_synthetic.txt.gz"
    df.to_csv(tmp, sep="\t", index=False, compression="gzip")

    got = stream_aggregate(tmp).set_index("gene_id")
    expected = df.assign(z2=df["Zscore"] ** 2).groupby("Gene")["z2"].agg(["sum", "count", "max"])

    ok = True
    for gene in expected.index:
        if gene not in got.index:
            ok = False
            print(f"[selftest] MISSING gene {gene}")
            continue
        row_got = got.loc[gene]
        row_exp = expected.loc[gene]
        if not (np.isclose(row_got["sum_z2"], row_exp["sum"]) and
                row_got["n_snps_tested"] == row_exp["count"] and
                np.isclose(row_got["max_z2"], row_exp["max"])):
            ok = False
            print(f"[selftest] MISMATCH gene {gene}: got={row_got.to_dict()} "
                  f"expected={row_exp.to_dict()}")
    print(f"[selftest] {'PASS' if ok else 'FAIL'} -- {len(expected)} genes, "
          f"{n:,} synthetic rows, chunked aggregation matches direct pandas groupby")
    assert ok, "self-test failed -- do not trust this script on the real file yet"


def main():
    if SMOKE_TEST:
        print("[mode] SMOKE_TEST=1 (default) -- running self-test only, NOT downloading "
              "the real 3.88GB file. Set PHASE_SMOKE_TEST=0 to run for real.")
        _selftest()
        return

    OUT_DIR.mkdir(parents=True, exist_ok=True)
    fetch()
    agg = stream_aggregate(LOCAL_GZ)
    agg.to_parquet(OUT_PARQUET, index=False)
    print(f"[done] wrote {OUT_PARQUET}")

    report = {
        "instrument": "eQTLGen FULL NOMINAL cis-eQTL summary statistics (every tested "
                      "cis-SNP-gene pair within 1Mb of gene center, >=2 cohorts) -- "
                      "per-gene sum(Zscore^2) over ALL tested SNPs, not just FDR<0.05 "
                      "significant ones. Genetics-only.",
        "n_genes": int(len(agg)),
        "n_rows_processed": int(agg["n_snps_tested"].sum()),
        "next_step": "Bring eqtlgen_full_nominal_cis_h2.parquet back into "
                     "data/processed/. Engineer reruns G0a-v2's primary per-gene "
                     "cross-cohort Spearman using sum_z2 from THIS file (full window) "
                     "instead of the significant-only aggregate, against the same "
                     "OneK1K score, to see whether the per-gene bar (0.5) is rescued.",
    }
    with open(OUT_REPORT, "w") as f:
        json.dump(report, f, indent=2)
    print(f"[done] wrote {OUT_REPORT}")


if __name__ == "__main__":
    main()